# Reproduce LGEA (EMNLP 2025) — DBP15K

*Breaking the Noise Barrier: LLM-Guided Semantic Filtering and Enhancement for Multi-Modal
Entity Alignment* · [paper](https://aclanthology.org/2025.emnlp-main.1684.pdf) ·
[code](https://github.com/alusang/LGEA-framework)

**Targets — Table 1:**

| split | Hits@1 | Hits@10 | MRR |
|---|---|---|---|
| ZH-EN | .970 | .998 | .982 |
| JA-EN | .962 | .999 | .977 |
| FR-EN | .975 | .999 | .985 |

---

### Before you hit run

| | |
|---|---|
| **Accelerator** | **GPU T4 ×2** — *not* P100. Llama-3-8B fp16 needs 16GB; one 15GB card is not enough, and Turing/Pascal have no bf16. |
| **Internet** | ON (for pip and the model weights; OFF is impossible even with a data bundle) |
| **How to run** | *Save Version → Save & Run All (Commit)*. 12h budget. |
| **Wall time** | ~2.1h per split (embed ~1.5h, train ~35min), plus ~15min download if no bundle |
| **Llama-3 is gated** | Accept the licence at [meta-llama/Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) and add your token as Kaggle Secret `HF_TOKEN`, **or** point `BASE_MODEL` at the ungated mirror in the config cell. |

Three splits is ~6.5h, which fits one session but leaves little slack. Running one split
per session is the safer path — every stage is resumable per split.

### Scope caveat — put this on your slide

The visual semantic filtering module (**contribution #1**) is **not** reproduced here. It
needs BLIP captions over the 50GB raw-image dump that is only distributed via Baidu Cloud.
`model/NewEA.py:269` makes `mask_path=None` fall through to an **all-ones mask**, which is
exactly the paper's own `0%mask` setting (Table 4) — a defined configuration, not a broken
run. This notebook **does** reproduce **contribution #2** (LLM attribute summarisation).

### Resume

Each stage writes a per-split marker into `/kaggle/working` and is skipped if already
done, so a re-run after a crash picks up where it stopped. Set a `FORCE_*` flag to redo one.

## 0 · Config

In [1]:
import os

# Which DBP15K splits to run. Each costs ~2.1h. Start with one.
SPLITS = ['ja_en', 'fr_en']                     # ['zh_en', 'ja_en', 'fr_en']

# Pre-built data as a Kaggle dataset (see make_kaggle_bundle.py). When set, the notebook
# copies from here and skips both the 1.4GB download and the attribute build. Leave as
# None to download from the original sources instead.
BUNDLE = None                          # e.g. '/kaggle/input/lgea-data'

DATA_RATE = '0.3'      # paper sec 5.1: 30% seed alignments for DBP15K
EPOCHS    = '500'      # --enable_sota clamps DBP15K to 250 in config.py:200
SEED      = '42'

# --- Llama-3-8B base weights ------------------------------------------------------
# The LLM2Vec repos ship a LoRA adapter ONLY (adapter_model.safetensors + config +
# tokenizer, no weight shards), so the base has to be loaded explicitly.
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'          # gated: needs HF_TOKEN secret
# BASE_MODEL = 'NousResearch/Meta-Llama-3-8B-Instruct'      # ungated mirror, same weights
L2V_MNTP   = 'McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp'
L2V_SUP    = 'McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised'

# PIN THE REMOTE CODE. On 2025-12-02 the McGill repo rewrote modeling_llama_encoder.py to
# require transformers>=4.56.2 (it imports transformers.modeling_layers). But llm2vec pins
# transformers>=4.43.1,<=4.44.2, so pip gives you 4.44.2 and main/ blows up with
#     ModuleNotFoundError: No module named 'transformers.modeling_layers'
# 1af5f3140c42 (2024-05-21) is the last revision written against the 4.44 API, and it is
# the one the paper's code was developed against.
CODE_REV = '1af5f3140c42'

# sum_embed.py:65 does `desc += entids[id] + ' ' + desc`, prepending the entity URI to the
# text -- while the paper says three times that surface forms are not used (sec 4.1, 5.1,
# Table 11) and model/NewEA.py loads files named att_sum_*_noname.pkl. We follow the paper.
INCLUDE_ENTITY_NAME = False

BATCH_SIZE = 16        # 32 OOMs a 15GB T4 at 512 tokens
MAX_LENGTH = 512       # same as sum_embed.py
MAX_ATTR_PER_ENTITY = 40

FORCE_DATA  = False
FORCE_EMBED = False
FORCE_TRAIN = False

# --- layout -----------------------------------------------------------------------
# config.py:13 does data_root = <repo>/../../data, so the tree must be exactly
#     ROOT/code/LGEA-framework/  and  ROOT/data/mmkg/
TEMP = '/kaggle/temp'                  # scratch: does NOT count against the 20GB output cap
WORK = '/kaggle/working'
ROOT = f'{WORK}/lgea'
CODE = f'{ROOT}/code/LGEA-framework'
DATA = f'{ROOT}/data/mmkg'
PKL_DIR = f'{DATA}/pkls'

def split_dir(s):  return f'{DATA}/DBP15K/{s}'
def img_pkl(s):    return f'{s}_GA_id_img_feature_dict.pkl'   # built by src/data.py:73
def raw_dir(s):    return f'{TEMP}/raw/DBP15k_raw/{s}'
def emb_path(side, s): return f'{WORK}/att_sum_{side}_noname_{s}.pkl'

def done(tag):  return os.path.exists(f'{WORK}/.done_{tag}')
def mark(tag):  open(f'{WORK}/.done_{tag}', 'w').close()

for d in (TEMP, ROOT, PKL_DIR, f'{DATA}/dump'):
    os.makedirs(d, exist_ok=True)
for s in SPLITS:
    os.makedirs(split_dir(s), exist_ok=True)

print('splits:', SPLITS, '| bundle:', BUNDLE or '(download)')

splits: ['ja_en', 'fr_en'] | bundle: (download)


## 1 · Install

The pins are not cosmetic: `llm2vec` requires `transformers>=4.43.1,<=4.44.2`, and that is
what forces `CODE_REV` above. Run this **before** anything imports `transformers` — a
downgrade does not take effect in a kernel that already imported it.

In [2]:
!pip install -q gdown easydict unidecode 2>&1 | tail -2
!pip install -q "llm2vec==0.2.3" "peft==0.12.0" "transformers==4.44.2" 2>&1 | tail -3

from importlib.metadata import version
for p in ('transformers', 'peft', 'llm2vec', 'torch', 'numpy', 'accelerate'):
    try:
        print(f'{p:14s} {version(p)}')
    except Exception as e:
        print(f'{p:14s} MISSING ({e})')
assert version('transformers').startswith('4.44'), \
    'transformers must be 4.44.x for CODE_REV 1af5f3140c42 -- see the config cell'

!nvidia-smi --query-gpu=name,memory.total --format=csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
transformers   4.44.2
peft           0.12.0
llm2vec        0.2.3
torch          2.10.0+cu128
numpy          2.0.2
accelerate     1.13.0
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 2 · Data

With `BUNDLE` set, everything is copied from the attached Kaggle dataset. Otherwise two
sources are fetched, because neither alone is enough:

* **MEAformer package (1.26G)** — DBP15K text + the ResNet image-feature pkls.
* **`HackCz/DBP15K_raw` (150M)** — raw attribute triples *with values*. The MEAformer
  package only ships attribute *names* (`training_attrs_*`), which cannot be summarised.

The LGEA repo ships neither, nor any of the preprocessed artifacts.

In [3]:
import subprocess, shutil, glob

if not os.path.exists(CODE):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/alusang/LGEA-framework.git', CODE], check=True)
    print('cloned LGEA-framework')

if BUNDLE:
    src_root = f'{BUNDLE}/mmkg'
    assert os.path.isdir(src_root), (
        f'{src_root} not found. Attach the dataset built by make_kaggle_bundle.py, '
        f'or set BUNDLE = None. /kaggle/input holds: {os.listdir("/kaggle/input")}')
    for s in SPLITS:
        shutil.copytree(f'{src_root}/DBP15K/{s}', split_dir(s), dirs_exist_ok=True)
        shutil.copy2(f'{src_root}/pkls/{img_pkl(s)}', f'{PKL_DIR}/{img_pkl(s)}')
        print(f'{s}: copied from bundle')
elif done('data') and not FORCE_DATA:
    print('data already prepared, skipping')
else:
    !gdown -q 1VIWcc3KDcLcRImeSrF2AyhetBLq_gsnx -O {TEMP}/mmkg.zip
    !mkdir -p {TEMP}/mea && unzip -q -o {TEMP}/mmkg.zip -d {TEMP}/mea
    !wget -q -O {TEMP}/DBP_raw.zip https://huggingface.co/datasets/HackCz/DBP15K_raw/resolve/main/DBP_raw.zip
    !unzip -q -o {TEMP}/DBP_raw.zip -d {TEMP}
    for s in SPLITS:
        !unzip -q -o {TEMP}/DBP15k_raw_all/DBP15k.zip 'DBP15k_raw/{s}/*' -d {TEMP}/raw

    def find_mmkg(root):
        '''The archive top-level folder name has changed between re-uploads; go find it.'''
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath) == 'mmkg' and 'DBP15K' in dirnames:
                return dirpath
        raise SystemExit(f'could not locate an "mmkg" directory under {root}')

    MMKG_SRC = find_mmkg(f'{TEMP}/mea')
    print('mmkg source:', MMKG_SRC)
    for s in SPLITS:
        for name in ['ent_ids_1', 'ent_ids_2', 'ill_ent_ids', 'triples_1', 'triples_2']:
            shutil.copy2(f'{MMKG_SRC}/DBP15K/{s}/{name}', f'{split_dir(s)}/{name}')
        shutil.copy2(f'{MMKG_SRC}/pkls/{img_pkl(s)}', f'{PKL_DIR}/{img_pkl(s)}')
        print(f'{s}: copied text files + {img_pkl(s)}')

cloned LGEA-framework
mmkg source: /kaggle/temp/mea/mmkg
ja_en: copied text files + ja_en_GA_id_img_feature_dict.pkl
fr_en: copied text files + fr_en_GA_id_img_feature_dict.pkl


## 3 · Build the three files the repo expects but does not ship

Skipped when a bundle supplied them.

| file | read by | format |
|---|---|---|
| `attribute1.txt` / `attribute2.txt` | `src/data.py::load_attr` (only checks `len(eval(line)) != 0` → `mask_a`) **and** the summarisation step (`';'.join(info)`) | `<id>\t['(pred, value)', ...]` — elements must be `str`, not tuples |
| `ent2id` | `model/NewEA.py:78`, which sizes the visual mask from it | `<id>\t<entity URI>` |

In [4]:
import re
from collections import defaultdict

# <subj> <pred> "value"^^<type> .   |   <subj> <pred> "value"@lang .
TRIPLE_RE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.*)\s+\.\s*$')
VALUE_RE  = re.compile(r'^"(.*)"(?:\^\^<[^>]+>|@[\w-]+)?$', re.DOTALL)

def short(uri):
    return uri.rsplit('/', 1)[-1].rsplit('#', 1)[-1]

def read_ent_ids(path):
    out = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line:
                eid, uri = line.split('\t')
                out[uri] = int(eid)
    return out

def parse_att_triples(path, uri2id, cap):
    per_ent, kept, skipped = defaultdict(list), 0, 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            m = TRIPLE_RE.match(line.rstrip('\n'))
            if not m:
                skipped += 1
                continue
            subj, pred, raw = m.groups()
            eid = uri2id.get(subj)
            if eid is None:
                continue
            vm = VALUE_RE.match(raw.strip())
            value = (vm.group(1) if vm else raw.strip()).replace('\\"', '"').replace('\n', ' ').strip()
            if not value or len(per_ent[eid]) >= cap:
                continue
            per_ent[eid].append(f'({short(pred)}, {value})')
            kept += 1
    return per_ent, kept, skipped

ENT_NUM = {}
for s in SPLITS:
    sd = split_dir(s)
    left_ids  = read_ent_ids(f'{sd}/ent_ids_1')   # ids MUST come from these files:
    right_ids = read_ent_ids(f'{sd}/ent_ids_2')   # the image pkl is indexed by them
    assert not (set(left_ids.values()) & set(right_ids.values())), f'{s}: entity ids collide'
    ENT_NUM[s] = len(left_ids) + len(right_ids)

    if os.path.exists(f'{sd}/attribute1.txt') and os.path.exists(f'{sd}/ent2id'):
        print(f'{s}: attribute files already present ({ENT_NUM[s]} entities)')
        continue

    src_lang = s.split('_')[0]
    for lang, uri2id, out_name in ((src_lang, left_ids, 'attribute1.txt'),
                                   ('en',     right_ids, 'attribute2.txt')):
        att_path = f'{raw_dir(s)}/{lang}_att_triples.txt'
        if not os.path.exists(att_path):                  # some splits ship it unsuffixed
            att_path = f'{raw_dir(s)}/{lang}_att_triples'
        # guard: the raw dump's own ent_ids must agree with MEAformer's, or every
        # attribute vector would be silently misaligned against the image features
        raw_ids = f'{raw_dir(s)}/ent_ids_{1 if out_name.endswith("1.txt") else 2}'
        if os.path.exists(raw_ids):
            assert read_ent_ids(raw_ids) == uri2id, f'{raw_ids} disagrees with MEAformer ids'

        per_ent, kept, skipped = parse_att_triples(att_path, uri2id, MAX_ATTR_PER_ENTITY)
        n_empty = 0
        with open(f'{sd}/{out_name}', 'w', encoding='utf-8') as f:
            for eid in sorted(uri2id.values()):
                items = per_ent.get(eid, [])
                n_empty += not items
                f.write(f'{eid}\t{items!r}\n')
        print(f'{s}/{out_name}: {len(per_ent)} with attrs, {n_empty} empty, '
              f'{kept} pairs kept, {skipped} unparsed lines')

    with open(f'{sd}/ent2id', 'w', encoding='utf-8') as f:
        for uri2id in (left_ids, right_ids):
            for uri, eid in sorted(uri2id.items(), key=lambda kv: kv[1]):
                f.write(f'{eid}\t{uri}\n')
    print(f'{s}/ent2id: {ENT_NUM[s]} lines')

if not BUNDLE:
    mark('data')

ja_en/attribute1.txt: 14999 with attrs, 4815 empty, 291819 pairs kept, 31 unparsed lines
ja_en/attribute2.txt: 14999 with attrs, 4781 empty, 350866 pairs kept, 684 unparsed lines
ja_en/ent2id: 39594 lines
fr_en/attribute1.txt: 14997 with attrs, 4664 empty, 323565 pairs kept, 193 unparsed lines
fr_en/attribute2.txt: 15000 with attrs, 4993 empty, 367914 pairs kept, 965 unparsed lines
fr_en/ent2id: 39654 lines


In [5]:
# --- verify before spending any GPU time ---
import pickle

for s in SPLITS:
    sd = split_dir(s)
    n_attr = sum(1 for n in ('attribute1.txt', 'attribute2.txt')
                   for _ in open(f'{sd}/{n}', encoding='utf-8'))
    n_e2i  = sum(1 for _ in open(f'{sd}/ent2id', encoding='utf-8'))
    with open(f'{PKL_DIR}/{img_pkl(s)}', 'rb') as f:
        img = pickle.load(f)

    ok = 0
    for n in ('attribute1.txt', 'attribute2.txt'):
        for line in open(f'{sd}/{n}', encoding='utf-8'):
            v = eval(line.rstrip('\n').split('\t')[1])       # exactly what load_attr does
            assert isinstance(v, list) and all(isinstance(x, str) for x in v)
            ok += 1

    print(f'{s}: ENT_NUM={ENT_NUM[s]}  ent2id={n_e2i}  attr_lines={n_attr} ({ok} eval-clean)  '
          f'images={len(img)} ({100*len(img)/ENT_NUM[s]:.1f}%, dim={len(next(iter(img.values())))})')
    assert n_e2i == ENT_NUM[s] == n_attr
    assert all(isinstance(k, int) for k in list(img)[:1000]), 'image pkl keys must be int ids'
print('\nSTAGE 1 OK')

ja_en: ENT_NUM=39594  ent2id=39594  attr_lines=39594 (39594 eval-clean)  images=26480 (66.9%, dim=2048)
fr_en: ENT_NUM=39654  ent2id=39654  attr_lines=39654 (39654 eval-clean)  images=28032 (70.7%, dim=2048)

STAGE 1 OK


## 4 · LLM attribute summarisation + embedding  *(contribution #2)*

Reimplements `sum_embed.py`. Forced changes:

1. **Base weights loaded explicitly** — the LLM2Vec repo has no weight shards.
2. **fp16, not bf16** — T4 is Turing.
3. **Explicit device map** — `'auto'` splits a decoder layer across both cards (see below).
4. **No entity name prepended** — `INCLUDE_ENTITY_NAME`.

Entities with zero attribute triples are *skipped*, not embedded from an empty string:
`load_attr` zero-fills any id absent from the pkl, and `mask_a` already flags them.

The 8B model is loaded **once** and reused for every split.

In [6]:
# Prompt is Table 11 of the paper, verbatim.
PROMPT = '''
You are an expert who can provide concise explanations based on entity information.
I will give you the properties of attributes and values of an entity in the form of (predicate object).
Using these information, please provide a short description of the entity.

- The explanation should be no longer than 100 words.
- Focus on summarizing the entity based on the given information.
- Do not include unnecessary details or explanations beyond the entity description.
- Do not include entity name.

Example:

Entity Information: (occupation Mathematician), (notable work Principia Mathematica)
Explanation: The entity was a prominent mathematician recognized for authoring Principia
Mathematica, a foundational work in the history of science and mathematics.

Now, please summarize the following entity information and return an description in English:
'''

def build_pairs(split, which):
    '''-> (ids, [[instruction, text], ...]) for attribute1.txt (1) or attribute2.txt (2).'''
    sd = split_dir(split)
    names = {}
    if INCLUDE_ENTITY_NAME:
        with open(f'{sd}/ent2id', encoding='utf-8') as f:
            for line in f:
                eid, nm = line.rstrip('\n').split('\t')
                names[eid] = nm
    ids, pairs, skipped = [], [], 0
    with open(f'{sd}/attribute{which}.txt', encoding='utf-8') as f:
        for line in f:
            eid, info = line.rstrip('\n').split('\t')
            items = eval(info)
            if not items:
                skipped += 1
                continue
            desc = ';'.join(items)
            if INCLUDE_ENTITY_NAME:
                desc = f'{names[eid]} {desc}'
            ids.append(eid)
            pairs.append([PROMPT, desc])
    print(f'{split}/attribute{which}.txt: {len(ids)} to embed, {skipped} zero-filled')
    return ids, pairs

pending = [s for s in SPLITS if FORCE_EMBED or not done(f'embed_{s}')]
print('\nsplits needing embedding:', pending or 'none')


splits needing embedding: ['ja_en', 'fr_en']


In [7]:
import torch, numpy as np, pickle, time, gc

if not pending:
    print('all embeddings already present, skipping the 8B model entirely')
else:
    from transformers import AutoTokenizer, AutoModel, AutoConfig
    from peft import PeftModel
    from llm2vec import LLM2Vec
    from tqdm.auto import trange

    if BASE_MODEL.startswith('meta-llama'):
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

    tokenizer = AutoTokenizer.from_pretrained(L2V_MNTP, revision=CODE_REV)
    config    = AutoConfig.from_pretrained(L2V_MNTP, trust_remote_code=True, revision=CODE_REV)

    # --- explicit device map: device_map='auto' SPLITS a decoder layer here ------------
    # transformers sets _no_split_modules = ["LlamaDecoderLayer"], but LlamaEncoderModel
    # builds ModifiedLlamaDecoderLayer. accelerate matches no_split by CLASS NAME, so the
    # subclass isn't protected and one layer gets scattered over both cards. Its attention
    # then runs on cuda:0 and its MLP on cuda:1, and the layer dies at
    #   hidden_states = residual + hidden_states
    #   RuntimeError: Expected all tensors to be on the same device, but found at least
    #                 two devices, cuda:0 and cuda:1!
    # LlamaEncoderModel.__init__ creates exactly embed_tokens / layers.N / norm (no
    # rotary_emb -- it predates that refactor), so these keys are complete.
    n_layers = config.num_hidden_layers
    layer_split = n_layers // 2 - 1     # cuda:0 also carries the ~1.0GB embedding
    device_map = {'embed_tokens': 0}
    device_map.update({f'layers.{i}': (0 if i < layer_split else 1) for i in range(n_layers)})
    device_map['norm'] = 1

    # code_revision (NOT revision) pins the downloaded modeling_llama_encoder.py:
    # config.auto_map is a cross-repo reference, so the code comes from the McGill repo
    # while `revision` here would only apply to BASE_MODEL's weights.
    model = AutoModel.from_pretrained(BASE_MODEL, trust_remote_code=True, config=config,
                                      code_revision=CODE_REV,
                                      torch_dtype=torch.float16, device_map=device_map,
                                      low_cpu_mem_usage=True)
    for i, layer in enumerate(model.layers):
        devs = {str(p.device) for p in layer.parameters()}
        assert len(devs) == 1, f'layer {i} is split across {devs}'
    print(f'{n_layers} layers intact; cuda:0 = embed + layers 0-{layer_split-1}, '
          f'cuda:1 = layers {layer_split}-{n_layers-1} + norm')

    model = PeftModel.from_pretrained(model, L2V_MNTP, revision=CODE_REV)
    model = model.merge_and_unload()
    model = PeftModel.from_pretrained(model, L2V_SUP)

    # --- realign the second adapter across the shards --------------------------------
    # peft places LoRA weights on the device of their base layer, but only at layer
    # construction time. L2V_SUP is loaded AFTER merge_and_unload() stripped accelerate's
    # hooks, so peft sees no hf_device_map and drops every adapter tensor on cuda:0.
    from peft.tuners.tuners_utils import BaseTunerLayer

    def adapter_tensors(module):
        for attr in list(module.adapter_layer_names) + list(module.other_param_names):
            d = getattr(module, attr, None)
            if hasattr(d, 'items'):
                for key, sub in d.items():
                    yield attr, key, sub

    n_adapters = 0
    for module in model.modules():
        if isinstance(module, BaseTunerLayer):
            n_adapters += 1
            for key in {k for _, k, _ in adapter_tensors(module)}:
                module._move_adapter_to_device_of_base_layer(key)   # peft's own helper

    mismatched = []
    for mname, module in model.named_modules():
        if not isinstance(module, BaseTunerLayer):
            continue
        base = getattr(module.get_base_layer(), 'weight', None)
        if base is None:
            continue
        for attr, key, sub in adapter_tensors(module):
            for p in getattr(sub, 'parameters', lambda: [])():
                if p.device != base.device:
                    mismatched.append(f'{mname}.{attr}.{key}: {p.device} vs {base.device}')
    print(f'realigned {n_adapters} adapter layers, {len(mismatched)} still mismatched')
    assert not mismatched, mismatched[:5]

    model.eval()
    l2v = LLM2Vec(model, tokenizer, pooling_mode='mean', max_length=MAX_LENGTH)

    def encode_sharded(pairs, desc):
        '''Replacement for l2v.encode() -- see llm2vec/llm2vec.py:357.

        With torch.cuda.device_count() > 1, encode() forks one process per GPU and pickles
        the model. That assumes DATA parallel (a full replica per card), which:
          (a) cannot pickle the closures a multi-GPU device_map installs, giving
              AttributeError: Can't get local object 'add_hook_to_module.<locals>.new_forward'
          (b) would not fit anyway -- 8B fp16 is ~16GB per replica vs a 15GB T4.
        We need PIPELINE parallel, so we keep the shard and run encode()'s single-process
        branch by hand -- same sort/restore order, but no self.to() call.
        '''
        texts = [l2v._convert_to_str(instr, txt) for instr, txt in pairs]
        l2v.eval()
        order = np.argsort([-l2v._text_length(t) for t in texts])   # longest first
        ordered = [texts[i] for i in order]
        dev = next(l2v.model.parameters()).device                   # first shard
        chunks = []
        for i in trange(0, len(ordered), BATCH_SIZE, desc=desc):
            feats = l2v.tokenize([l2v.prepare_for_tokenization(t)
                                  for t in ordered[i:i + BATCH_SIZE]])
            feats = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in feats.items()}
            with torch.no_grad():
                chunks.append(l2v.forward(feats).detach().cpu())
        return torch.cat(chunks, dim=0)[np.argsort(order)].to(torch.float32).numpy()

    # Smoke test on 2 texts: a device or shape bug shows up in seconds, not an hour in.
    probe = encode_sharded(build_pairs(pending[0], 1)[1][:2], 'smoke')
    assert probe.shape == (2, 4096) and np.isfinite(probe).all() and np.linalg.norm(probe[0]) > 0
    print(f'smoke test ok: {probe.shape}, norm={np.linalg.norm(probe[0]):.3f}')

    for s in pending:
        t0 = time.time()
        for side, which in (('left', 1), ('right', 2)):
            ids, pairs = build_pairs(s, which)
            emb = encode_sharded(pairs, f'{s}:{side}')
            with open(emb_path(side, s), 'wb') as f:
                pickle.dump({eid: emb[i] for i, eid in enumerate(ids)}, f)
            print(f'{s} {side}: {emb.shape} -> {emb_path(side, s)}')
        mark(f'embed_{s}')
        print(f'{s} embedded in {(time.time()-t0)/60:.1f} min\n')

    del l2v, model, tokenizer, config
    gc.collect(); torch.cuda.empty_cache()

print('GPU free:', [f'{torch.cuda.mem_get_info(i)[0]/1e9:.1f}G'
                    for i in range(torch.cuda.device_count())])

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

modeling_llama_encoder.py: 0.00B [00:00, ?B/s]

...

ja_en right: (14999, 4096) -> /kaggle/working/att_sum_right_noname_ja_en.pkl
ja_en embedded in 174.0 min

fr_en/attribute1.txt: 14997 to embed, 4664 zero-filled


fr_en:left:   0%|          | 0/938 [00:00<?, ?it/s]

fr_en left: (14997, 4096) -> /kaggle/working/att_sum_left_noname_fr_en.pkl
fr_en/attribute2.txt: 15000 to embed, 4993 zero-filled


fr_en:right:   0%|          | 0/938 [00:00<?, ?it/s]

fr_en right: (15000, 4096) -> /kaggle/working/att_sum_right_noname_fr_en.pkl
fr_en embedded in 175.3 min

GPU free: ['15.5G', '15.0G']


In [8]:
# --- sanity check: do the embeddings carry any alignment signal at all? ---
# Near zero would mean something is wrong (random weights, misaligned ids) and there is
# no point spending hours on training.
for s in SPLITS:
    with open(emb_path('left', s), 'rb') as f:  la = pickle.load(f)
    with open(emb_path('right', s), 'rb') as f: ra = pickle.load(f)
    lk, rk = list(la), list(ra)
    le = torch.nn.functional.normalize(torch.tensor(np.array([la[k] for k in lk])), p=2, dim=1)
    re_= torch.nn.functional.normalize(torch.tensor(np.array([ra[k] for k in rk])), p=2, dim=1)
    sim = (le @ re_.T).numpy()

    gt = {}
    with open(f'{split_dir(s)}/ill_ent_ids', encoding='utf-8') as f:
        for line in f:
            a, b = line.strip().split('\t')
            gt[a] = b
    rpos = {k: i for i, k in enumerate(rk)}

    h1 = h10 = mrr = n = 0
    for i, lid in enumerate(lk):
        t = rpos.get(gt.get(lid))
        if t is None:
            continue
        rank = int(np.where(np.argsort(-sim[i]) == t)[0][0]) + 1
        h1 += rank == 1; h10 += rank <= 10; mrr += 1/rank; n += 1
    print(f'{s}: attribute-only retrieval over {n} gold pairs -> '
          f'Hits@1={h1/n:.3f}  Hits@10={h10/n:.3f}  MRR={mrr/n:.3f}')
    del sim, le, re_

print('\nNOT the paper number -- a lower bound with no graph structure, no images, no '
      'training. Note the attribute triples themselves contain (name, <surface form>) '
      'pairs, which is why this is already so high.')
print('STAGE 2 OK')

ja_en: attribute-only retrieval over 14998 gold pairs -> Hits@1=0.845  Hits@10=0.939  MRR=0.880
fr_en: attribute-only retrieval over 14997 gold pairs -> Hits@1=0.853  Hits@10=0.952  MRR=0.890

NOT the paper number -- a lower bound with no graph structure, no images, no training. Note the attribute triples themselves contain (name, <surface form>) pairs, which is why this is already so high.
STAGE 2 OK


## 5 · Train

Mirrors `run_newea.sh`, except:

* `--workers 2` — Kaggle has 4 CPU cores, the script assumes 12
* `CUDA_VISIBLE_DEVICES=0` — single-GPU run (`dist=0` anyway)
* `--mask_path` omitted → `None` → all-ones mask (the `0%mask` configuration)
* one patch to `model/layers.py`, below — a memory fix, not a model change

### Patch: the GAT sparse-attention backward allocates an N×N dense matrix

`SpecialSpmmFunction.backward` (`model/layers.py`) does

```python
grad_a_dense = grad_output.matmul(b.t())              # N x N
edge_idx     = a._indices()[0,:] * ctx.N + a._indices()[1,:]
grad_values  = grad_a_dense.view(-1)[edge_idx]        # ...then keeps only E entries
```

At N = 38,960 that intermediate is 38960² × 4 B = **5.66 GiB**, allocated twice per
attention head (`layers.py:86` and `:90`). On the paper's A100-40GB it fits; on a 15GB T4
it is exactly a `Tried to allocate 5.66 GiB` OOM.

`grad_a_dense[i,j]` is just `sum_k grad_output[i,k] * b[j,k]`, so the E entries actually
used can be computed edge-wise in O(E·d) instead of O(N²) — **the same numbers**, verified
against the original formulation before shipping this. The accumulation is fp32 so
autocast's fp16 costs no precision relative to the cuBLAS matmul it replaces.

In [9]:
LAYERS = f'{CODE}/model/layers.py'
ORIGINAL = """            grad_a_dense = grad_output.matmul(b.t())
            edge_idx = a._indices()[0, :] * ctx.N + a._indices()[1, :]
            grad_values = grad_a_dense.view(-1)[edge_idx]"""

PATCHED = """            # PATCHED (memory only -- numerically identical).
            # The original built the full N x N dense product grad_output @ b.t() and
            # then gathered E entries from it; at N=38,960 that is a 5.66 GiB tensor and
            # OOMs a 15GB T4. grad_a_dense[i, j] == sum_k grad_output[i, k] * b[j, k],
            # so compute only the E entries we need, chunked, accumulating in fp32.
            _row = a._indices()[0, :]
            _col = a._indices()[1, :]
            _E = _row.numel()
            _step = max(1, int(2e7) // max(1, b.size(1)))
            grad_values = torch.empty(_E, device=grad_output.device, dtype=grad_output.dtype)
            for _s in range(0, _E, _step):
                _e = min(_s + _step, _E)
                grad_values[_s:_e] = (
                    grad_output.index_select(0, _row[_s:_e]).float()
                    * b.index_select(0, _col[_s:_e]).float()
                ).sum(dim=1).to(grad_output.dtype)"""

source = open(LAYERS, encoding='utf-8').read()
if 'PATCHED (memory only' in source:
    print('layers.py already patched')
else:
    assert source.count(ORIGINAL) == 1, (
        'layers.py does not contain the expected backward() body -- upstream changed, '
        're-read model/layers.py before patching')
    open(LAYERS, 'w', encoding='utf-8').write(source.replace(ORIGINAL, PATCHED))
    print('patched SpecialSpmmFunction.backward in model/layers.py')

recheck = open(LAYERS, encoding='utf-8').read()
# Check the executable line, not the name "grad_a_dense" -- the comment mentions it.
assert 'grad_output.matmul(b.t())' not in recheck, 'the N x N allocation is still there'
import ast; ast.parse(recheck)
print('layers.py parses clean, no N x N allocation remains')

patched SpecialSpmmFunction.backward in model/layers.py
layers.py parses clean, no N x N allocation remains


In [10]:
for s in SPLITS:
    sd = split_dir(s)
    for side in ('left', 'right'):
        shutil.copy2(emb_path(side, s), f'{sd}/att_sum_{side}_noname.pkl')

    required = ['ent_ids_1', 'ent_ids_2', 'ill_ent_ids', 'triples_1', 'triples_2',
                'attribute1.txt', 'attribute2.txt', 'ent2id',
                'att_sum_left_noname.pkl', 'att_sum_right_noname.pkl']
    missing = [n for n in required if not os.path.exists(f'{sd}/{n}')]
    assert not missing, f'{s}: MISSING {missing}'
    assert os.path.exists(f'{PKL_DIR}/{img_pkl(s)}')

    with open(f'{sd}/att_sum_left_noname.pkl', 'rb') as f:  la = pickle.load(f)
    with open(f'{sd}/att_sum_right_noname.pkl', 'rb') as f: ra = pickle.load(f)
    covered = {int(k) for k in la} | {int(k) for k in ra}
    all_ids = set()
    for n in ('ent_ids_1', 'ent_ids_2'):
        with open(f'{sd}/{n}', encoding='utf-8') as f:
            all_ids |= {int(l.split('\t')[0]) for l in f if l.strip()}
    # ~77% is expected: the raw dump only has attribute triples for the linked entities.
    assert covered <= all_ids, f'{s}: attribute pkl holds ids not in ent_ids'
    assert len(covered) > 0.7 * len(all_ids), f'{s}: low attribute coverage {len(covered)}'
    print(f'{s}: preflight OK  dim={len(next(iter(la.values())))}  '
          f'coverage {len(covered)}/{len(all_ids)} ({100*len(covered)/len(all_ids):.1f}%)')

ja_en: preflight OK  dim=4096  coverage 29998/39594 (75.8%)
fr_en: preflight OK  dim=4096  coverage 29997/39654 (75.6%)


In [11]:
# expandable_segments: an earlier failing run held 10.59 GB with only 4.65 GB live --
# 5.81 GB was reserved-but-unallocated fragmentation.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

for s in SPLITS:
    if done(f'train_{s}') and not FORCE_TRAIN:
        print(f'{s}: already trained, skipping')
        continue
    cmd = ['python', 'main.py',
           '--gpu', '0', '--eval_epoch', '1', '--only_test', '0',
           '--model_name', 'NewEA', '--data_choice', 'DBP15K',
           '--data_split', s, '--data_rate', DATA_RATE,
           '--epoch', EPOCHS, '--lr', '5e-4',               # lr: paper Table 7 best
           '--hidden_units', '300,300,300',
           '--save_model', '0', '--batch_size', '3500',
           '--csls', '--csls_k', '3', '--random_seed', SEED,
           '--exp_name', 'LGEA_repro', '--exp_id', f'v1_{s}_{DATA_RATE}',
           '--workers', '2', '--dist', '0', '--accumulation_steps', '1',
           '--scheduler', 'cos',
           '--attr_dim', '300', '--img_dim', '300',
           '--name_dim', '300', '--char_dim', '300', '--hidden_size', '300',
           '--tau', '0.1', '--structure_encoder', 'gat',
           '--num_attention_heads', '1', '--num_hidden_layers', '1',
           '--use_surface', '0',                            # paper sec 5.1: no surface forms
           '--use_intermediate', '1', '--enable_sota']
    print('\n' + ' '.join(cmd) + '\n')
    proc = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:          # stream, so a long commit still shows progress
        print(line, end='')
    proc.wait()
    assert proc.returncode == 0, f'{s}: training exited {proc.returncode}'
    mark(f'train_{s}')


python main.py --gpu 0 --eval_epoch 1 --only_test 0 --model_name NewEA --data_choice DBP15K --data_split ja_en --data_rate 0.3 --epoch 500 --lr 5e-4 --hidden_units 300,300,300 --save_model 0 --batch_size 3500 --csls --csls_k 3 --random_seed 42 --exp_name LGEA_repro --exp_id v1_ja_en_0.3 --workers 2 --dist 0 --accumulation_steps 1 --scheduler cos --attr_dim 300 --img_dim 300 --name_dim 300 --char_dim 300 --hidden_size 300 --tau 0.1 --structure_encoder gat --num_attention_heads 1 --num_hidden_layers 1 --use_surface 0 --use_intermediate 1 --enable_sota

INFO - 09/26/26 08:22:05 - 0:00:00 - ============ Initialized logger ============
INFO - 09/26/26 08:22:05 - 0:00:00 - The experiment will be stored in /kaggle/working/lgea/data/mmkg/dump/0926-LGEA_repro/NewEA_DBP15K_ja_en_v1_ja_en_0.3
                                     
INFO - 09/26/26 08:22:05 - 0:00:00 - Running command: python main.py --gpu 0 --eval_epoch 1 --only_test 0 --model_name NewEA --data_choice DBP15K --data_split ja_en --d

## 6 · Results

The log does not print the string `hits@1`. It writes

```
Ep 249 | l2r: acc of top [1, 10, 50] = [0.9578 0.9974 1.    ], mr = 1.117, mrr = 0.975
```

so the parser reads that form, from the final block after `--------- Test result ---------`,
and reports both directions (`l2r` = left-to-right, `r2l` = right-to-left).

In [12]:
import glob, re

PAPER = {'zh_en': (.970, .998, .982),
         'ja_en': (.962, .999, .977),
         'fr_en': (.975, .999, .985)}
ROW = re.compile(r'(l2r|r2l): acc of top \[1, 10, 50\] = \[([\d.\s]+)\].*?mrr = ([\d.]+)')

print(f'{"split":8s} {"dir":4s} {"Hits@1":>16s} {"Hits@10":>16s} {"MRR":>16s}')
print(f'{"":8s} {"":4s} {"paper / ours":>16s} {"paper / ours":>16s} {"paper / ours":>16s}')
for s in SPLITS:
    logs = sorted(glob.glob(f'{DATA}/dump/**/*{s}*/train.log', recursive=True),
                  key=os.path.getmtime)
    if not logs:
        print(f'{s:8s} no log found'); continue
    text = open(logs[-1], errors='ignore').read()
    if 'Test result' not in text:
        print(f'{s:8s} log has no final test block (run did not finish)'); continue
    block = text[text.rindex('Test result'):]
    p1, p10, pm = PAPER[s]
    for direction, accs, mrr in ROW.findall(block):
        a = [float(x) for x in accs.split()]
        print(f'{s:8s} {direction:4s} {p1:6.3f} / {a[0]:6.4f} {p10:6.3f} / {a[1]:6.4f} '
              f'{pm:6.3f} / {float(mrr):6.3f}')

print('\nDifferences to declare: no visual mask (= paper 0%mask, Table 4); '
      f'INCLUDE_ENTITY_NAME={INCLUDE_ENTITY_NAME}; attribute coverage limited to the '
      '~77% of entities present in the raw dump.')

split    dir            Hits@1          Hits@10              MRR
                  paper / ours     paper / ours     paper / ours
ja_en    l2r   0.962 / 0.9518  0.999 / 0.9957  0.977 /  0.969
ja_en    r2l   0.962 / 0.9490  0.999 / 0.9958  0.977 /  0.968
fr_en    l2r   0.975 / 0.9501  0.999 / 0.9963  0.985 /  0.968
fr_en    r2l   0.975 / 0.9516  0.999 / 0.9962  0.985 /  0.969

Differences to declare: no visual mask (= paper 0%mask, Table 4); INCLUDE_ENTITY_NAME=False; attribute coverage limited to the ~77% of entities present in the raw dump.
